## Load Data
 Load all the data of Olist to local DuckDB DB

In [1]:
import duckdb
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
con = duckdb.connect("../data/olist.duckdb")

In [2]:
for f in RAW.glob("*.csv"):
    con.execute(f"CREATE OR REPLACE TABLE {f.stem} AS SELECT * FROM read_csv_auto('{f}')")
    print(f"Loaded {f.stem} into DuckDB")
# con.close

Loaded olist_customers_dataset into DuckDB
Loaded olist_geolocation_dataset into DuckDB
Loaded olist_orders_dataset into DuckDB
Loaded olist_order_items_dataset into DuckDB
Loaded olist_order_payments_dataset into DuckDB
Loaded olist_order_reviews_dataset into DuckDB
Loaded olist_products_dataset into DuckDB
Loaded olist_sellers_dataset into DuckDB
Loaded product_category_name_translation into DuckDB


In [3]:
## Checking the data loaded into DuckDB
for f in RAW.glob("*.csv"):
    print(f.stem, con.execute(f"SELECT COUNT(*) FROM {f.stem}").fetchone()[0])

olist_customers_dataset 99441
olist_geolocation_dataset 1000163
olist_orders_dataset 99441
olist_order_items_dataset 112650
olist_order_payments_dataset 103886
olist_order_reviews_dataset 99224
olist_products_dataset 32951
olist_sellers_dataset 3095
product_category_name_translation 71


## Data Overview

In [4]:
# row and column count for every table
con.execute("""
    select table_name, estimated_size as rows, column_count as columns
    from duckdb_tables()
    order by table_name
""").df()

,table_name,rows,columns
0,olist_customers_dataset,99441,5
1,olist_geolocation_dataset,1000163,5
2,olist_order_items_dataset,112650,7
3,olist_order_payments_dataset,103886,5
4,olist_order_reviews_dataset,99224,7
5,olist_orders_dataset,99441,8
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


In [5]:
# number of orders per month
monthly = con.execute("""
    select strftime(order_purchase_timestamp, '%Y-%m') as month, count(*) as orders
    from olist_orders_dataset
    group by month
    order by month
""").df()

monthly

,month,orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


Range of Data: September 2016 till October 2018.
In total, there 99,441 orders placed in this time frame.

In [6]:
# columns with missing values, across all tables
tables = [t[0] for t in con.execute("select table_name from duckdb_tables() order by table_name").fetchall()]

missing = []
for t in tables:
    s = con.execute(f"summarize {t}").df()
    s = s[s.null_percentage > 0][["column_name", "null_percentage"]]
    s.insert(0, "table", t)
    missing.append(s)

pd.concat(missing, ignore_index=True)

,table,column_name,null_percentage
0,olist_order_reviews_dataset,review_comment_title,88.34
1,olist_order_reviews_dataset,review_comment_message,58.70
2,olist_orders_dataset,order_approved_at,0.16
3,olist_orders_dataset,order_delivered_carrier_date,1.79
4,olist_orders_dataset,order_delivered_customer_date,2.98
5,olist_products_dataset,product_category_name,1.85
6,olist_products_dataset,product_name_lenght,1.85
7,olist_products_dataset,product_description_lenght,1.85
8,olist_products_dataset,product_photos_qty,1.85
9,olist_products_dataset,product_weight_g,0.01


## Table Summaries

### 1. Customer Table

In [7]:
summarise_customers_table = con.execute("summarize olist_customers_dataset").df()
summarise_customers_table

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,customer_id,VARCHAR,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99,97769,None,None,None,None,None,99441,0.0
1,customer_unique_id,VARCHAR,0000366f3b9a7992bf8c76cfdf3221e2,ffffd2657e2aad2907e67c3e9daecbeb,98358,None,None,None,None,None,99441,0.0
2,customer_zip_code_prefix,VARCHAR,01003,99990,15086,None,None,None,None,None,99441,0.0
3,customer_city,VARCHAR,abadia dos dourados,zortea,4186,None,None,None,None,None,99441,0.0
4,customer_state,VARCHAR,AC,TO,26,None,None,None,None,None,99441,0.0


There are no null values in customers table

In [8]:
# unique users
unique_users = con.execute("""
    select count(distinct customer_id) as customer_ids,
           count(distinct customer_unique_id) as unique_people
    from olist_customers_dataset
""")
unique_users.df()

,customer_ids,unique_people
0,99441,96096


Number of unique users are 96,096 and have placed in total 99441 orders. A new customer id is created for each order.

### 2. Orders Table

In [9]:
summarise_orders_table = con.execute("summarize olist_orders_dataset").df()
summarise_orders_table

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,99974,NaN,None,NaN,NaN,NaN,99441,0.00
1,customer_id,VARCHAR,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99,97769,NaN,None,NaN,NaN,NaN,99441,0.00
2,order_status,VARCHAR,approved,unavailable,8,NaN,None,NaN,NaN,NaN,99441,0.00
3,order_purchase_timestamp,TIMESTAMP,2016-09-04 21:15:19,2018-10-17 17:30:18,96571,2017-12-31 08:43:12.776581,None,2017-09-12 09:46:15.721942,2018-01-19 08:25:16.131472,2018-05-04 21:22:52.465787,99441,0.00
4,order_approved_at,TIMESTAMP,2016-09-15 12:16:38,2018-09-03 17:40:06,87279,2017-12-31 18:35:24.0988,None,2017-09-12 23:38:47.244113,2018-01-19 19:31:11.91185,2018-05-05 10:39:06.448894,99441,0.16
5,order_delivered_carrier_date,TIMESTAMP,2016-10-08 10:34:01,2018-09-11 19:48:28,81859,2018-01-04 21:49:48.138278,None,2017-09-17 05:01:14.896067,2018-01-24 13:03:07.097667,2018-05-08 03:14:38.982959,99441,1.79
6,order_delivered_customer_date,TIMESTAMP,2016-10-11 13:46:32,2018-10-17 13:22:46,93526,2018-01-14 12:09:19.035543,None,2017-09-25 19:42:45.079512,2018-02-03 08:46:48.115237,2018-05-16 04:30:00.070728,99441,2.98
7,order_estimated_delivery_date,TIMESTAMP,2016-09-30 00:00:00,2018-11-12 00:00:00,451,2018-01-24 03:08:37.730111,None,2017-10-03 07:03:45.764487,2018-02-14 12:08:43.90428,2018-05-26 07:48:29.412612,99441,0.00


There are null values in order_delivered_customer_date, order_approved_at, and order_delivered_carrier_date. Mostly the ones without dates are likely in process or cancelled.

In [10]:
delivery_date_missing = con.execute("""
    select order_status, count(*) as missing_delivery_date_count from olist_orders_dataset
    where order_delivered_customer_date is null
    group by order_status
    order by missing_delivery_date_count desc
""").df()
delivery_date_missing

,order_status,missing_delivery_date_count
0,shipped,1107
1,canceled,619
2,unavailable,609
3,invoiced,314
4,processing,301
5,delivered,8
6,created,5
7,approved,2


Only 8 orders that are marked as delivered are missing delivery date, whereas rest all are genuine cases of in transit, or cancelled or haven't reached conclusion.

In [11]:
# Order count per user distribution
users = con.execute("""
    with order_per_customer as (select customer_unique_id, count(*) as order_count
    from olist_customers_dataset
    group by customer_unique_id)
    select order_count, count(*) as users, round(count(*) * 100.0 / (select count(*) from order_per_customer), 2) as percentage
    from order_per_customer
    group by order_count
    order by order_count asc
""")
users.df()

,order_count,users,percentage
0,1,93099,96.88
1,2,2745,2.86
2,3,203,0.21
3,4,30,0.03
4,5,8,0.01
5,6,6,0.01
6,7,3,0.00
7,9,1,0.00
8,17,1,0.00


~97% of users have placed only single order. Which is a huge number meaning business has very few repeat customers. 

In [12]:
# Order status distribution
order_status = con.execute("""
    select order_status, count(*) as orders, round(count(*) * 100.0 / (select count(*) from olist_orders_dataset), 2) as percentage
    from olist_orders_dataset
    group by order_status
    order by orders desc
""")
order_status.df()

,order_status,orders,percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


### 3. Order Items Data

In [13]:
summarise_order_items_table = con.execute("select * from olist_order_items_dataset").df()
summarise_order_items_table.describe()


,order_item_id,shipping_limit_date,price,freight_value
count,112650.000000,112650,112650.000000,112650.000000
mean,1.197834,2018-01-07 15:36:52.192685,120.653739,19.990320
min,1.000000,2016-09-19 00:15:34,0.850000,0.000000
25%,1.000000,2017-09-20 20:57:27.500000,39.900000,13.080000
50%,1.000000,2018-01-26 13:59:35,74.990000,16.260000
75%,1.000000,2018-05-10 14:34:00.750000,134.900000,21.150000
max,21.000000,2020-04-09 22:35:08,6735.000000,409.680000
std,0.705124,NaN,183.633928,15.806405


There is a shipping limit date for 2020, whereas the data is from 2016 till 2018. This looks like an anamoly. Will be investigated further.

In [14]:
summarise_order_items_table.info()
summarise_order_items_table.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


np.int64(0)

There are no null values in the order items table, and no duplicates. The table has 112,650 rows and 7 columns.

In [15]:
summarise_order_items_table[summarise_order_items_table.shipping_limit_date>'2018-12-31']

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
8643,13bdf405f961a6deec817d817f5c6624,1,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51,69.99,14.66
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22,75.99,14.70
85729,c2bb89b5c1dd978d507284be78a04cb2,1,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44
85730,c2bb89b5c1dd978d507284be78a04cb2,2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44


In total, there are 4 orders that have shipping date limit greater that 2018, most likely its a data entry error. We can exclude them from modelling data.

### Payments Data

In [16]:
summarise_payments_table = con.execute(""" summarize olist_order_payments_dataset""").df()
summarise_payments_table

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,99974,NaN,NaN,NaN,NaN,NaN,103886,0.0
1,payment_sequential,BIGINT,1,29,29,1.0926785129853878,0.7065837791949948,1,1,1,103886,0.0
2,payment_type,VARCHAR,boleto,voucher,5,NaN,NaN,NaN,NaN,NaN,103886,0.0
3,payment_installments,BIGINT,0,24,24,2.853348863176944,2.687050673856486,1,1,4,103886,0.0
4,payment_value,DOUBLE,0.0,13664.08,29627,154.10038041698365,217.49406386472384,56.77380184130726,100.10889896724083,171.75934799732582,103886,0.0


In [17]:
# payment rows that look invalid
con.execute("""
    select count(*) filter (where payment_installments = 0) as zero_installments,
           count(*) filter (where payment_value = 0) as zero_value,
           count(*) filter (where payment_type = 'not_defined') as type_not_defined
    from olist_order_payments_dataset
""").df()

,zero_installments,zero_value,type_not_defined
0,2,9,3


No missing values. A few rows look invalid (2 with zero installments, 9 with zero value, 3 with type 'not_defined'), too few to matter. payment_sequential goes up to 29 because some orders are paid in several parts(multiple vouchers), so payments will need to be summed per order before joining to avoid duplicating orders.

### Reviews Data

In [18]:
summarise_reviews_table = con.execute(""" summarize olist_order_reviews_dataset""").df()
summarise_reviews_table

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,review_id,VARCHAR,0001239bc1de2e33cb583967c2ca4c67,fffefe7a48d22f7b32046421062219d1,97803,NaN,NaN,NaN,NaN,NaN,99224,0.00
1,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,99168,NaN,NaN,NaN,NaN,NaN,99224,0.00
2,review_score,BIGINT,1,5,5,4.08642062404257,1.3475791311150984,4,5,5,99224,0.00
3,review_comment_title,VARCHAR,,🔟,4538,NaN,NaN,NaN,NaN,NaN,99224,88.34
4,review_comment_message,VARCHAR,\r\n,😡😡😡😡😡👎👎👎👎👎\r\nEmpresa sem compromisso com o cl...,36834,NaN,NaN,NaN,NaN,NaN,99224,58.70
5,review_creation_date,TIMESTAMP,2016-10-02 00:00:00,2018-08-31 00:00:00,623,2018-01-12 20:49:23.948238,NaN,2017-09-23 17:25:10.479927,2018-02-02 01:41:34.499906,2018-05-15 13:38:27.73651,99224,0.00
6,review_answer_timestamp,TIMESTAMP,2016-10-07 18:32:28,2018-10-29 12:27:35,101131,2018-01-16 00:23:56.977939,NaN,2017-09-27 04:24:14.096925,2018-02-05 01:25:10.310572,2018-05-19 02:24:29.965388,99224,0.00


Missing values are present in comment message and comment title. These are not mandatory fields and usually customers leave it out so it's totally normal and does not need any treatment.

In [19]:
# number and share of reviews per star score
scores = con.execute("""
    select review_score, count(*) as reviews,
           round(count(*) * 100.0 / (select count(*) from olist_order_reviews_dataset), 1) as percentage
    from olist_order_reviews_dataset
    group by review_score
    order by review_score
""").df()
scores

,review_score,reviews,percentage
0,1,11424,11.5
1,2,3151,3.2
2,3,8179,8.2
3,4,19142,19.3
4,5,57328,57.8


There are 14.7% of reviews scored as 1 and 2. These are defined as bad reviews, 3 is a neutral and 4,5 are good reviews which is 77.1% of the overall distribution.

In [20]:
# orders with more than one review
con.execute("""
    select count(*) as orders_with_multiple_reviews
    from (select order_id from olist_order_reviews_dataset group by order_id having count(*) > 1)
""").df()

,orders_with_multiple_reviews
0,547


547 orders have more than one review. The earliest review per order will be kept.

In [21]:
# score distribution keeping only the earliest review per order
con.execute("""
    select review_score, count(*) as reviews,
           round(count(*) * 100.0 / sum(count(*)) over (), 1) as percentage
    from (
        select order_id, review_score
        from olist_order_reviews_dataset
        qualify row_number() over (partition by order_id order by review_answer_timestamp) = 1
    )
    group by review_score
    order by review_score
""").df()

,review_score,reviews,percentage
0,1,11346,11.5
1,2,3130,3.2
2,3,8120,8.2
3,4,19053,19.3
4,5,57024,57.8


When considered only 1 review per order, the distbrution is unchanged, the lower rating is still 14.7%(rated 1 and 2)

### Products Data

In [18]:
con.execute("select * from olist_products_dataset").df().duplicated().sum()


np.int64(0)

There are no duplicate products in the data

In [19]:
summarise_products_table = con.execute(""" summarize olist_products_dataset""").df()
summarise_products_table

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,product_id,VARCHAR,00066f42aeeb9f3007548bb9d3f33c38,fffe9eeff12fcbd74a2f2b007dde0c58,32523,NaN,NaN,NaN,NaN,NaN,32951,0.00
1,product_category_name,VARCHAR,agro_industria_e_comercio,utilidades_domesticas,71,NaN,NaN,NaN,NaN,NaN,32951,1.85
2,product_name_lenght,BIGINT,5,76,66,48.47694876472589,10.245740725237287,42,51,57,32951,1.85
3,product_description_lenght,BIGINT,4,3992,2902,771.4952846232337,635.1152246349538,339,594,973,32951,1.85
4,product_photos_qty,BIGINT,1,20,19,2.1889861166939797,1.7367656379315435,1,1,3,32951,1.85
5,product_weight_g,BIGINT,0,40425,2234,2276.4724877841513,4282.038730977024,300,701,1883,32951,0.01
6,product_length_cm,BIGINT,7,105,100,30.81507784758263,16.914458054065953,18,25,39,32951,0.01
7,product_height_cm,BIGINT,2,105,103,16.937661234028347,13.637554061749569,8,13,21,32951,0.01
8,product_width_cm,BIGINT,6,118,95,23.196728277034204,12.079047453227794,15,20,30,32951,0.01


Mostly the products with 1.85% missing data are missing all the details like category name, name, description and photos quantity. There are also products with weight as 0.

In [20]:
missing_product_info = con.execute("""
    select * from olist_products_dataset
    where product_category_name is null or product_category_name = '' or product_weight_g is null or product_length_cm is null or product_height_cm is null or product_width_cm is null or product_weight_g =0
""").df()
missing_product_info.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [21]:
missing_product_info[missing_product_info.product_category_name.isnull()].isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [22]:
missing_product_info[missing_product_info.product_weight_g.isnull()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
166,09ff539a621711667c43eba6a3bd8466,bebes,60,865,3,<NA>,<NA>,<NA>,<NA>
369,5eb564652db742ff8f28759cd8d2652a,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [23]:
missing_product_info[missing_product_info.product_weight_g==0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
184,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51,529,1,0,30,25,30
264,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48,528,1,0,30,25,30
289,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53,528,1,0,30,25,30
598,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53,528,1,0,30,25,30


The 610 products with no category name are also missing name length, description length and photo count. One of them is also missing weight and dimensions. In addition, 4 products have a weight of zero.

The 4 zero-weight products are left as they are, since they make up only 0.01% of products and won't affect the analysis.

### Category Translation Data

In [24]:
summarise_categories_translation_table = con.execute(""" summarize product_category_name_translation""").df()
summarise_categories_translation_table

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,product_category_name,VARCHAR,agro_industria_e_comercio,utilidades_domesticas,69,None,None,None,None,None,71,0.0
1,product_category_name_english,VARCHAR,agro_industry_and_commerce,watches_gifts,73,None,None,None,None,None,71,0.0


In [25]:
con.execute("select * from product_category_name_translation").df().duplicated().sum()

np.int64(0)

In [26]:
# Checking which product categories are missing translations
missing_translations = con.execute("""
    select p.product_category_name, count(distinct p.product_id) as products, count(distinct i.order_id) as affected_orders
    from olist_products_dataset p   
    left join product_category_name_translation t
    on p.product_category_name = t.product_category_name
    left join olist_order_items_dataset i
    on p.product_id = i.product_id
    where t.product_category_name is null and p.product_category_name is not null
    group by p.product_category_name
""").df()
missing_translations

,product_category_name,products,affected_orders
0,portateis_cozinha_e_preparadores_de_alimentos,10,14
1,pc_gamer,3,8


2 categories used by products have no English translation, these cover 13 products and 22 orders. In the feature table, these keep their Portuguese name where no English name exists.

### Sellers Data

In [27]:
sellers_table = con.execute(""" select * from olist_sellers_dataset""").df()
sellers_table.describe()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
count,3095,3095,3095,3095
unique,3095,2246,611,23
top,3442f8959a84dea7ee197c632cb2df15,14940,sao paulo,SP
freq,1,49,694,1849


In [28]:
print( sellers_table.info())
print("Sellers Table Null Values:", sellers_table.isnull().sum())
print("Sellers Table Duplicates:", sellers_table.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   str  
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: str(4)
memory usage: 96.8 KB
None
Sellers Table Null Values: seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64
Sellers Table Duplicates: 0


There are no missing values and no duplciate data.

### Geolocation Data

In [29]:
geo_table = con.execute(""" select * from olist_geolocation_dataset""").df()
geo_table.describe()

,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


In [30]:
print( geo_table.info())
print("Geolocation Table Null Values:", geo_table.isnull().sum())
print("Geolocation Table Duplicates:", geo_table.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  str    
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), str(3)
memory usage: 38.2 MB
None
Geolocation Table Null Values: geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
Geolocation Table Duplicates: 261831


There are in total 261831 duplicate entries in the geolocation data

### Delivery vs reviews

In [42]:
# one row per order: order dates, earliest review, delivery status and delivery time
subset_data = con.execute("""
    select o.order_id, o.order_status, o.order_purchase_timestamp, o.order_approved_at,
           o.order_delivered_carrier_date, o.order_delivered_customer_date, o.order_estimated_delivery_date,
           r.review_score, r.review_answer_timestamp,
          case when r.review_score <= 2 then 1 else 0 end as bad_review,
           case when o.order_delivered_customer_date is null then null
                when o.order_delivered_customer_date > o.order_estimated_delivery_date then 'late'
                else 'on time' end as delivery_status
    from olist_orders_dataset o
    join olist_order_reviews_dataset r on o.order_id = r.order_id
    qualify row_number() over (partition by o.order_id order by r.review_answer_timestamp) = 1
""").df()


In [43]:
subset_data.shape

(98673, 11)

In [52]:
# subset_data.order_status.value_counts()

In [44]:
subset_data.head()

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_answer_timestamp,bad_review,delivery_status
0,af5b0604e113e1526593d505a26ef39d,delivered,2017-09-27 12:07:22,2017-09-27 12:26:02,2017-09-29 22:25:42,2017-09-30 17:13:01,2017-10-16,1,2017-10-03 23:54:54,1,on time
1,c158e9806f85a33877bdfd4f607b72e7,delivered,2017-04-14 22:06:32,2017-04-14 22:15:13,2017-04-17 14:52:32,2017-05-08 11:10:26,2017-05-18,4,2017-05-10 17:34:48,0,on time
2,33209e0f87b2ed543a0876812c9c5c40,delivered,2018-04-11 14:59:05,2018-04-11 15:12:22,2018-04-12 18:49:47,2018-04-13 19:04:19,2018-04-27,3,2018-04-15 00:58:44,0,on time
3,18d686f1bb1270182c646fe337459d79,delivered,2018-02-15 15:07:11,2018-02-15 15:27:08,2018-02-17 00:14:57,2018-03-11 19:28:52,2018-03-07,1,2018-03-09 10:11:39,1,late
4,4cd9478b63edacb631dcf780b036b157,delivered,2018-06-23 19:29:12,2018-06-26 05:12:58,2018-06-27 09:06:00,2018-06-30 13:50:47,2018-07-23,4,2018-07-03 22:47:44,0,on time


In [45]:
# average score and bad review rate for late vs on time deliveries
subset_data.groupby("delivery_status").agg(
    orders=("order_id", "count"),
    avg_score=("review_score", "mean"),
    bad_review_pct=("bad_review", "mean"),
).round(3)

,orders,avg_score,bad_review_pct
delivery_status,,,
late,7662,2.568,0.540
on time,88168,4.294,0.092


Late orders get far worse reviews: their average score is 2.6 against 4.3 for on-time orders, and 54% of them get a 1-2 star review compared with 9% of on-time orders. Late delivery affects 8% of delivered orders (7,662 of 95,830).

In [61]:
# days delivered after the promised date (negative = early)
subset_data["days_vs_promise"] = (subset_data.order_delivered_customer_date - subset_data.order_estimated_delivery_date).dt.days

# group into bands from early to late
bands = [-1000, -7, 0, 3, 7, 1000]
labels = ["7+ days early", "0-6 days early", "1-3 days late", "4-7 days late", "8+ days late"]
subset_data["vs_promise"] = pd.cut(subset_data.days_vs_promise, bins=bands, labels=labels)

# orders, share of orders, average score and bad review rate per band
by_band = subset_data.groupby("vs_promise").agg(
    orders=("order_id", "count"),
    avg_score=("review_score", "mean"),
    bad_review_pct=("bad_review", "mean"),
)
by_band["pct_of_orders"] = by_band.orders / by_band.orders.sum() * 100
by_band["bad_review_pct"] = by_band.bad_review_pct * 100

by_band[["orders", "pct_of_orders", "avg_score", "bad_review_pct"]].round(1).reset_index()

,vs_promise,orders,pct_of_orders,avg_score,bad_review_pct
0,7+ days early,75749,79.0,4.3,9.0
1,0-6 days early,13699,14.3,4.2,10.7
2,1-3 days late,1852,1.9,3.3,32.2
3,4-7 days late,1748,1.8,2.1,67.5
4,8+ days late,2782,2.9,1.7,79.2


~93% of orders are delivered early than the promised date. These get 9-10% of bad reviews. Whereas, if the order is late by 1-3 days, the bad reviews jump by roughly 22 percentage points and adds to 32%. and if orders are later by more than 3 days, 68-79% will get bad reviews. Delivery timing is a main factor for poor reviews.

In [50]:
# latest shipping deadline per order
shipping_limit = con.execute("""
    select order_id, max(shipping_limit_date) as shipping_limit
    from olist_order_items_dataset
    group by order_id
""").df()
subset_data = subset_data.merge(shipping_limit, on="order_id", how="left")

# delivered orders flagged by whether the seller dispatched after the deadline
dispatched = subset_data[subset_data.order_delivered_carrier_date.notna() & subset_data.delivery_status.notna()].copy()


In [57]:
dispatched.delivery_status.value_counts()

delivery_status
on time    88168
late        7661
Name: count, dtype: int64

In [63]:
dispatched["dispatch"] = (dispatched.order_delivered_carrier_date > dispatched.shipping_limit).map({True: "seller late", False: "seller on time"})
dispatched["late"] = (dispatched.delivery_status == "late").astype(int)

# bad review and late delivery rates by dispatch timing
by_dispatch = dispatched.groupby("dispatch").agg(
    orders=("order_id", "count"),
    bad_review_pct=("bad_review", "mean"),
    late_delivery_pct=("late", "mean"),
).reset_index()
by_dispatch[["bad_review_pct", "late_delivery_pct"]] *= 100
by_dispatch["pct_of_orders"] = by_dispatch.orders / by_dispatch.orders.sum() * 100
by_dispatch.round(2)

,dispatch,orders,bad_review_pct,late_delivery_pct,pct_of_orders
0,seller late,8611,23.78,24.14,8.99
1,seller on time,87218,11.72,6.40,91.01


~9% of orders were handed to the carrier after the seller's shipping deadline. These orders get a bad review twice as often (24% vs 12%) and are almost four times as likely to arrive late (24% vs 6%). Since dispatch timing is known as soon as the parcel reaches the carrier, it's an early warning signal the business can act on before the customer receives the order.